# 07 · Signature demonstration

Pick Hog Rider → three legal decks, each with archetype, estimated win rate (with sample size and uncertainty), top synergy pairs with evidence, and advisory warnings. Then swap a card and watch the score and warnings recompute.

Estimated win rates describe top-player battles in the sampled window. They are not a promise about your games, and the top result of a search tends to be optimistic.

In [ ]:
import os
import json
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

DATA_SOURCE = os.environ.get("CRDECK_DATA_SOURCE", "real")  # "real" (default) or "synthetic"
from crdeck.config import Paths
paths = Paths.for_source(DATA_SOURCE)
print(paths.banner())

In [ ]:
from crdeck.pipeline import load_context
ctx = load_context(DATA_SOURCE)  # raises NoDataError with instructions if data is missing; never substitutes
print(ctx.paths.banner())
ctx.split.describe()

In [ ]:
from crdeck.search import Searcher, parse_seed
from crdeck.explain import build_explanation, render_text, explanations_frame
searcher = Searcher(ctx)
seeds = [parse_seed("Hog Rider", ctx.cards)]
results = searcher.complete(seeds)
explanations = [build_explanation(d.entries, ctx, searcher.objective_dict(d)) for d in results]
explanations_frame(explanations)

In [ ]:
for i, x in enumerate(explanations, 1):
    print(f"=== Deck {i} ===")
    print(render_text(x))
    print()

## Swap a card
Take the best suggested swap for deck 1 and recompute everything.

In [ ]:
from crdeck.rules import parse_deck_key
deck = results[0]
swaps = searcher.swaps(deck.entries, locked=seeds)
best = max(swaps, key=lambda s_: s_["delta_objective"])
print(f"Swap {best['replace']} -> {best['with']}: objective {best['delta_objective']:+.4f}, win prob {best['delta_win_prob']:+.4f}; {best['reason']}")
new_entries = parse_deck_key(best["new_deck_key"])
before, after = explanations[0], build_explanation(new_entries, ctx, searcher.objective_dict(searcher.evaluate(new_entries)))
print()
print(render_text(after))

In [ ]:
pd.DataFrame({
    "before": [before.score["meta_win_prob"], before.score["interval"], "; ".join(g["root_cause"] for g in before.warnings) or "none"],
    "after": [after.score["meta_win_prob"], after.score["interval"], "; ".join(g["root_cause"] for g in after.warnings) or "none"],
}, index=["meta win prob", "interval", "warnings"])

## Full explanation object
Everything above comes from this structure (spec §8).

In [ ]:
json.loads(json.dumps(explanations[0].to_dict(), default=str))